# Unified ERASE + SISA — Operator-Level FLOPs Evaluation

**Papers**:
- ERASE: *Fast Exact Unlearning for In-Context Learning Data for LLMs* (Muresanu et al., ICML 2025)
- SISA: *Machine Unlearning* (Bourtoule et al., 2021)

---

## What this notebook does

Computes the **Holistic Unlearning Cost** (Definition 4.1) for both **ERASE-{1,2,3,4}-shot**
and **{1,2,3,4}-SISA** in a single unified experiment, using:

- **Dataset**: `jinzhuoran/RWKU` (Real-World Knowledge Unlearning)
  - Knowledge Base / Forget Corpus: `train_original_passage` split
  - Evaluation Queries: `forget_level1` split
  - Alignment: matched by the shared `subject` field
- **Model**: `meta-llama/Llama-3.1-8B`
- **Profiler**: `torch.utils.flop_counter.FlopCounterMode` — accurate, operator-level FLOP counts

$$C(M) = \frac{U_M - U_{1\text{-SISA}}}{I_{1\text{-SISA}} - I_M}$$

- $U_M$ = unlearning FLOP cost per request for method $M$
- $I_M$ = inference FLOP cost per query for method $M$
- Higher $C(M)$ = better (M can handle more inferences per unlearning request
  before becoming more expensive than 1-SISA)

### Method Summary

| Method | Unlearning Strategy | Inference Strategy |
|--------|--------------------|---------|
| **ERASE-k** | Quantized k-means index update, O(d) expected | k ICL examples prepended to query |
| **n-SISA** | Retrain affected shard (~1/n of data) | Aggregate n shard models |

### Key Design: Real ERASE Refit Detection

For ERASE unlearning, we **actually run** the quantized k-means unlearning and
record the **real path taken**:
- **FREE**: point was not the selected example, centroid unchanged → O(d)
- **SWAP**: point was selected, centroid unchanged → pick next closest, O(d)
- **REFIT**: centroid changed after removal → full refit, O(n·k·d·iter), rare

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 2: Install Dependencies
# ═══════════════════════════════════════════════════════════════════

!pip install -q transformers datasets sentence-transformers accelerate

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 3: Imports & Configuration
#
#  ALL tuneable parameters live here.  Change CONFIG values to swap
#  model, dataset, or profiling settings.
# ═══════════════════════════════════════════════════════════════════

import json
import time
import copy
import math
import warnings
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────
#  CONFIG — Edit these values to customise the experiment
# ─────────────────────────────────────────────────────────────────
CONFIG = {
    # ═══ MODEL ══════════════════════════════════════════════════
    "model_name":   "meta-llama/Llama-3.1-8B",
    "hf_token":     None,           # Set your HF token here for gated models
    "torch_dtype":  "bfloat16",     # "float16" | "bfloat16"

    # ═══ DATA ═══════════════════════════════════════════════════
    "dataset_name":       "jinzhuoran/RWKU",
    "passage_split":      "train_original_passage",   # Knowledge Base
    "query_split":        "forget_level1",            # Evaluation Queries
    "max_seq_length":     128,        # tokeniser truncation length (reduced for GPU memory)

    # ─── ERASE parameters (paper defaults) ─────────────────────
    "epsilon":         0.05,          # quantization granularity
    "max_kmeans_iter": 10,            # k-means iterations
    "encoder_name":    "sentence-transformers/all-MiniLM-L6-v2",
    "k_shot_list":     [1, 2, 3, 4],  # ERASE variants to evaluate

    # ─── SISA parameters ───────────────────────────────────────
    "n_shards_list":       [1, 2, 3, 4],  # SISA variants to evaluate
    "n_slices_per_shard":  2,              # slices per shard (checkpointing)

    # ─── FLOP profiling ────────────────────────────────────────
    "n_profile_steps":     5,         # training steps to profile
    "profile_batch_size":  1,         # batch size for profiling
    "n_epochs":            1,         # training epochs (for scaling)
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 4: Load Model — meta-llama/Llama-3.1-8B
# ═══════════════════════════════════════════════════════════════════

from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading model: {CONFIG['model_name']} ...")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    token=CONFIG["hf_token"],
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    token=CONFIG["hf_token"],
    torch_dtype=getattr(torch, CONFIG["torch_dtype"]),
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"\n{'='*60}")
print(f"  Model loaded: {CONFIG['model_name']}")
print(f"  Parameters:   {n_params:,}")
print(f"  Dtype:        {CONFIG['torch_dtype']}")
print(f"  Device:       {next(model.parameters()).device}")
print(f"{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 5: Load & Align RWKU Data
#
#  Knowledge Base:      train_original_passage  (text, subject)
#  Evaluation Queries:  forget_level1           (subject, query, answer, ...)
#  Alignment:           matched by shared 'subject' field
# ═══════════════════════════════════════════════════════════════════

from datasets import load_dataset

print(f"Loading Knowledge Base: {CONFIG['dataset_name']} / {CONFIG['passage_split']} ...")
passages_ds = load_dataset(
    CONFIG["dataset_name"],
    CONFIG["passage_split"],
)
# The passage split loads as a DatasetDict with a 'train' key
if hasattr(passages_ds, 'keys'):
    # DatasetDict — grab the first available split
    split_key = list(passages_ds.keys())[0]
    passages_ds = passages_ds[split_key]
print(f"  Passages loaded: {len(passages_ds)} rows")
print(f"  Features: {list(passages_ds.features.keys())}")

print(f"\nLoading Evaluation Queries: {CONFIG['dataset_name']} / {CONFIG['query_split']} ...")
queries_ds = load_dataset(
    CONFIG["dataset_name"],
    CONFIG["query_split"],
)
if hasattr(queries_ds, 'keys'):
    split_key = list(queries_ds.keys())[0]
    queries_ds = queries_ds[split_key]
print(f"  Queries loaded: {len(queries_ds)} rows")
print(f"  Features: {list(queries_ds.features.keys())}")

# ─── Build subject → passages mapping ──────────────────────────
subject_to_passages = defaultdict(list)
for i in range(len(passages_ds)):
    row = passages_ds[i]
    subject_to_passages[row["subject"]].append({
        "idx": i,
        "text": row["text"],
    })

# ─── Build aligned (query, passages) pairs ─────────────────────
aligned_data = []
for i in range(len(queries_ds)):
    row = queries_ds[i]
    subj = row["subject"]
    if subj in subject_to_passages:
        aligned_data.append({
            "subject": subj,
            "query": row["query"],
            "answer": row["answer"],
            "passage_indices": [p["idx"] for p in subject_to_passages[subj]],
        })

# ─── Extract flat lists for profiling ──────────────────────────
all_passages = passages_ds["text"]          # Full knowledge base
all_queries  = [d["query"] for d in aligned_data]  # Aligned queries
subjects     = list(subject_to_passages.keys())

print(f"\n{'='*60}")
print(f"  Data Alignment Summary")
print(f"  Total passages (Knowledge Base):  {len(all_passages):,}")
print(f"  Total aligned queries:            {len(all_queries):,}")
print(f"  Unique subjects:                  {len(subjects)}")
print(f"  Avg passages per subject:         {len(all_passages)/len(subjects):.1f}")
print(f"  Avg queries per subject:          {len(all_queries)/len(subjects):.1f}")
print(f"\n  Sample subject: '{subjects[0]}'")
print(f"  Sample passage (100 chars): {all_passages[0][:100]}...")
print(f"  Sample query: {all_queries[0]}")
print(f"{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 6: Core Classes
#
#  All algorithm implementations defined inline.
#  - UnlearningResult       (dataclass)
#  - QuantizedKMeans        (Ginart et al. 2019)
#  - ERASE                  (Algorithm 1, Muresanu et al. 2025)
#  - SISADataManager        (shard/slice partitioning)
#  - SISAUnlearner          (checkpoint-based cost estimation)
#  - SISAEnsemble           (multi-shard inference)
#  - HolisticCostEvaluator  (Definition 4.1)
# ═══════════════════════════════════════════════════════════════════

from sentence_transformers import SentenceTransformer


# ── Data Class ──────────────────────────────────────────────────

@dataclass
class UnlearningResult:
    """
    Complete record of one exact unlearning operation.

    Fields
    ------
    idx               : original index in the dataset that was unlearned
    cluster_id        : which cluster the point belonged to
    refit_needed      : True -> full k-means refit was required (expensive path)
    centroid_changed  : True -> the quantized centroid changed
    was_selected      : True -> this point was the chosen ICL example
    new_selected_text : replacement ICL example (when was_selected=True)
    unlearning_flops  : estimated FLOPS for this operation
    wall_time_s       : wall-clock seconds
    """
    idx: int
    cluster_id: int
    refit_needed: bool
    centroid_changed: bool
    was_selected: bool
    new_selected_text: Optional[str]
    unlearning_flops: int
    wall_time_s: float

    def __str__(self) -> str:
        path = "REFIT" if self.refit_needed else (
            "SWAP" if self.was_selected else "FREE"
        )
        return (
            f"UnlearningResult(idx={self.idx}, cluster={self.cluster_id}, "
            f"path={path}, flops={self.unlearning_flops:,}, "
            f"time={self.wall_time_s*1000:.1f}ms)"
        )


# ── Quantized K-Means (Ginart et al. 2019) ─────────────────────

class QuantizedKMeans:
    """
    Quantized K-Means with dataset-size-independent exact unlearning.

    Core idea: snap centroids to an epsilon-lattice so that removing
    a single point usually does NOT change the quantized centroid.
    When it doesn't change -> O(d) unlearning (cheap path).
    When it does change   -> full refit, but P(change) = O(1/(eps*|D|)).
    """

    def __init__(self, n_clusters, epsilon=0.05, max_iter=10, random_state=42):
        self.n_clusters   = n_clusters
        self.epsilon      = epsilon
        self.max_iter     = max_iter
        self.random_state = random_state

        self.data_               = None
        self.original_indices_   = None
        self.raw_centroids_      = None
        self.quantized_centroids_ = None
        self.labels_             = None
        self.theta_              = None
        self._rng = np.random.default_rng(random_state)

    def fit(self, X, original_indices=None):
        """Fit quantized k-means to embedding matrix X  (n, d)."""
        n, d = X.shape
        if n < self.n_clusters:
            raise ValueError(f"Need >= {self.n_clusters} samples, got {n}.")

        self.data_ = X.copy().astype(np.float64)
        self.original_indices_ = (
            np.arange(n) if original_indices is None
            else np.asarray(original_indices, dtype=int)
        )

        if self.theta_ is None:
            self.theta_ = self._rng.uniform(
                -self.epsilon / 2, self.epsilon / 2, d
            )

        self.raw_centroids_ = self._kmeanspp_init(self.data_)
        self.quantized_centroids_ = None

        for _ in range(self.max_iter):
            q_c = np.array([self._quantize(c) for c in self.raw_centroids_])
            dists = self._pairwise_l2(self.data_, q_c)
            new_labels = np.argmin(dists, axis=1)

            new_raw = np.zeros_like(self.raw_centroids_)
            for k in range(self.n_clusters):
                mask = new_labels == k
                if mask.sum() > 0:
                    new_raw[k] = self.data_[mask].mean(axis=0)
                else:
                    new_raw[k] = self.data_[
                        dists.min(axis=1).argmax()
                    ]

            new_q = np.array([self._quantize(c) for c in new_raw])
            if (
                self.quantized_centroids_ is not None
                and np.allclose(new_q, self.quantized_centroids_, atol=1e-12)
            ):
                self.labels_ = new_labels
                self.raw_centroids_ = new_raw
                break

            self.labels_ = new_labels
            self.raw_centroids_ = new_raw
            self.quantized_centroids_ = new_q

        self.quantized_centroids_ = np.array(
            [self._quantize(c) for c in self.raw_centroids_]
        )
        return self

    def unlearn(self, original_idx):
        """
        Remove one data point and decide whether a refit is needed.
        Returns (centroid_changed, refit_needed).
        """
        rows = np.where(self.original_indices_ == original_idx)[0]
        if len(rows) == 0:
            raise ValueError(f"original_idx={original_idx} not found.")
        row = int(rows[0])
        cluster_id = int(self.labels_[row])
        n_c = int((self.labels_ == cluster_id).sum())

        if n_c <= 1:
            self._remove_row(row)
            if len(self.data_) >= self.n_clusters:
                self.fit(self.data_, self.original_indices_)
            return True, True

        x_i = self.data_[row]
        new_raw = (n_c * self.raw_centroids_[cluster_id] - x_i) / (n_c - 1)
        new_q   = self._quantize(new_raw)
        centroid_changed = not np.allclose(
            new_q, self.quantized_centroids_[cluster_id], atol=1e-12
        )

        self._remove_row(row)

        if not centroid_changed:
            self.raw_centroids_[cluster_id] = new_raw
            return False, False
        else:
            if len(self.data_) >= self.n_clusters:
                self.fit(self.data_, self.original_indices_)
            return True, True

    def get_sorted_cluster(self, cluster_id):
        """Members of cluster_id sorted by distance to centroid."""
        mask = self.labels_ == cluster_id
        rows = np.where(mask)[0]
        if len(rows) == 0:
            return []
        centroid = self.quantized_centroids_[cluster_id]
        dists = np.linalg.norm(self.data_[rows] - centroid, axis=1)
        order = np.argsort(dists)
        return [
            (int(self.original_indices_[rows[i]]), float(dists[i]))
            for i in order
        ]

    def cluster_of(self, original_idx):
        rows = np.where(self.original_indices_ == original_idx)[0]
        if len(rows) == 0:
            raise ValueError(f"original_idx={original_idx} not found.")
        return int(self.labels_[rows[0]])

    @property
    def n_active(self):
        return len(self.data_) if self.data_ is not None else 0

    # ── Private helpers ───────────────────────────────────────
    def _quantize(self, centroid):
        return (
            np.round((centroid - self.theta_) / self.epsilon) * self.epsilon
            + self.theta_
        )

    def _kmeanspp_init(self, X):
        n = len(X)
        idx = int(self._rng.integers(0, n))
        centers = [X[idx].copy()]
        for _ in range(1, self.n_clusters):
            dists = np.array([
                min(np.linalg.norm(x - c) ** 2 for c in centers)
                for x in X
            ])
            probs = dists / dists.sum()
            idx = int(self._rng.choice(n, p=probs))
            centers.append(X[idx].copy())
        return np.array(centers)

    @staticmethod
    def _pairwise_l2(A, B):
        A2 = (A ** 2).sum(axis=1, keepdims=True)
        B2 = (B ** 2).sum(axis=1, keepdims=True).T
        AB = A @ B.T
        return np.sqrt(np.clip(A2 + B2 - 2 * AB, 0.0, None))

    def _remove_row(self, row):
        keep = np.ones(len(self.data_), dtype=bool)
        keep[row] = False
        self.data_             = self.data_[keep]
        self.original_indices_ = self.original_indices_[keep]
        self.labels_           = self.labels_[keep]


# ── ERASE Algorithm (Algorithm 1) ──────────────────────────────

class ERASE:
    """
    ERASE: Efficient Removal And Selection of Examples.

    1. Encode examples with Sentence-BERT.
    2. Run quantized k-means (k = n_examples).
    3. Select the closest example to each centroid as the ICL example.

    On unlearning (REAL path detection — not always assuming refit):
      - FREE: point was not selected, centroid unchanged -> O(d)
      - SWAP: point was selected, centroid unchanged -> pick next closest, O(d)
      - REFIT: centroid changed -> full refit, O(n*k*d*iter), rare
    """

    def __init__(
        self,
        n_examples=4,
        epsilon=0.05,
        encoder_name="sentence-transformers/all-MiniLM-L6-v2",
        max_iter=10,
        random_state=42,
    ):
        self.n_examples   = n_examples
        self.epsilon      = epsilon
        self.encoder_name = encoder_name
        self.max_iter     = max_iter
        self.random_state = random_state

        self._encoder = None
        self._qkm = QuantizedKMeans(
            n_clusters=n_examples,
            epsilon=epsilon,
            max_iter=max_iter,
            random_state=random_state,
        )

        self.texts_     = []
        self.metadata_  = []
        self.embeddings_ = None
        self.selected_  = {}   # cluster_id -> original_idx
        self._log       = []

    def fit(self, texts, metadata=None, precomputed_embeddings=None):
        """
        Encode texts and cluster with quantized k-means.

        Parameters
        ----------
        texts : list of str
        metadata : optional list of dicts
        precomputed_embeddings : (n, d) array — skip encoding if provided
        """
        if len(texts) < self.n_examples:
            raise ValueError(
                f"Need >= {self.n_examples} texts, got {len(texts)}."
            )
        self.texts_    = list(texts)
        self.metadata_ = metadata or [{} for _ in texts]

        if precomputed_embeddings is not None:
            self.embeddings_ = np.asarray(
                precomputed_embeddings, dtype=np.float64
            )
            print(f"[ERASE] Using {len(texts)} precomputed embeddings "
                  f"(d={self.embeddings_.shape[1]}).")
        else:
            if self._encoder is None:
                print(f"[ERASE] Loading encoder: {self.encoder_name}")
                self._encoder = SentenceTransformer(self.encoder_name)
            print(f"[ERASE] Encoding {len(texts)} examples...")
            self.embeddings_ = self._encoder.encode(
                texts, show_progress_bar=True, convert_to_numpy=True
            ).astype(np.float64)

        print(f"[ERASE] Running quantized k-means "
              f"(k={self.n_examples}, eps={self.epsilon})...")
        self._qkm.fit(
            self.embeddings_,
            original_indices=np.arange(len(texts)),
        )
        self._refresh_selected()
        print(f"[ERASE] Ready. Selected {len(self.selected_)} examples.")
        return self

    def get_selected_examples(self):
        """Return the k currently selected ICL examples."""
        out = []
        for cid in sorted(self.selected_):
            orig_idx = self.selected_[cid]
            text = self.texts_[orig_idx]
            if not text:
                continue
            sm = self._qkm.get_sorted_cluster(cid)
            dist = next((d for i, d in sm if i == orig_idx), float("nan"))
            out.append({
                "idx": orig_idx, "text": text,
                "cluster": cid, "distance_to_centroid": dist,
                "metadata": self.metadata_[orig_idx],
            })
        return out

    def get_icl_prompt(
        self, query, system_msg="Answer using only the context provided.",
    ):
        """
        Build a few-shot ICL prompt using Llama-3.1 chat template.

        Format:
          <|begin_of_text|><|start_header_id|>system<|end_header_id|>
          {system_msg}<|eot_id|>
          <|start_header_id|>user<|end_header_id|>
          {context examples + query}<|eot_id|>
          <|start_header_id|>assistant<|end_header_id|>
        """
        examples = self.get_selected_examples()
        few_shot = "\n".join(f"Context: {ex['text']}" for ex in examples)
        return (
            f"<|begin_of_text|>"
            f"<|start_header_id|>system<|end_header_id|>\n\n"
            f"{system_msg}<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n\n"
            f"{few_shot}\n\nQuestion: {query}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        )

    def unlearn(self, original_idx):
        """
        Exactly unlearn the example at original_idx.

        REAL path detection — checks whether refit is actually needed
        (not assumed). Three possible paths:
          1. FREE — not the selected example, centroid unchanged. O(d).
          2. SWAP — was selected, centroid unchanged. Pick next closest. O(d).
          3. REFIT — centroid changed. Full refit. O(n*k*d*iter). Rare.

        Returns UnlearningResult with the actual path and FLOP count.
        """
        t0 = time.perf_counter()
        if original_idx >= len(self.texts_) or not self.texts_[original_idx]:
            raise ValueError(f"original_idx={original_idx} invalid.")

        cluster_id   = self._qkm.cluster_of(original_idx)
        was_selected = (self.selected_.get(cluster_id) == original_idx)

        # ── Core operation: delegate to QuantizedKMeans ─────────
        # This is where the REAL check happens:
        #   - Computes new centroid after removing the point
        #   - Snaps to lattice
        #   - Compares with old quantized centroid
        #   - Only refits if centroid ACTUALLY changed
        centroid_changed, refit_needed = self._qkm.unlearn(original_idx)
        self.texts_[original_idx] = ""

        new_selected_text = None
        if refit_needed:
            # REFIT path: full k-means refit already happened
            self._refresh_selected()
        elif was_selected and not centroid_changed:
            # SWAP path: centroid is same, just pick next closest
            sm = self._qkm.get_sorted_cluster(cluster_id)
            valid = [(i, d) for i, d in sm if self.texts_[i]]
            if valid:
                new_idx = valid[0][0]
                self.selected_[cluster_id] = new_idx
                new_selected_text = self.texts_[new_idx]
            else:
                self._qkm.fit(self._qkm.data_, self._qkm.original_indices_)
                self._refresh_selected()
                refit_needed = True
        # else: FREE path — selected_ unchanged, nothing to do

        # ── Record actual FLOP cost based on real path ─────────
        d = self.embeddings_.shape[1]
        n_active = sum(1 for t in self.texts_ if t)
        if not refit_needed:
            # FREE or SWAP: O(d) — subtract + quantize + compare
            flops = 3 * d
        else:
            # REFIT: O(n * k * d * iter) — full k-means pass
            flops = n_active * self.n_examples * d * self.max_iter * 2

        result = UnlearningResult(
            idx=original_idx, cluster_id=cluster_id,
            refit_needed=refit_needed,
            centroid_changed=centroid_changed,
            was_selected=was_selected,
            new_selected_text=new_selected_text,
            unlearning_flops=flops,
            wall_time_s=time.perf_counter() - t0,
        )
        self._log.append(result)
        return result

    def _refresh_selected(self):
        self.selected_ = {}
        for cid in range(self.n_examples):
            sm = self._qkm.get_sorted_cluster(cid)
            valid = [(i, d) for i, d in sm if self.texts_[i]]
            if valid:
                self.selected_[cid] = valid[0][0]


# ── SISA Data Manager ──────────────────────────────────────────

class SISADataManager:
    """
    Partitions data into disjoint shards, each further divided into
    sequential slices for checkpoint-based unlearning.
    """

    def __init__(self, texts: list, n_shards: int, n_slices: int = 2):
        self.texts     = list(texts)
        self.n_total   = len(texts)
        self.n_shards  = n_shards
        self.n_slices  = n_slices

        self.shard_indices = []
        self.slice_indices = []

        base_size = self.n_total // n_shards
        remainder = self.n_total % n_shards

        offset = 0
        for s in range(n_shards):
            shard_size = base_size + (1 if s < remainder else 0)
            shard_idx = list(range(offset, offset + shard_size))
            self.shard_indices.append(shard_idx)

            slice_base = len(shard_idx) // n_slices
            slice_rem  = len(shard_idx) % n_slices
            slices = []
            s_off = 0
            for sl in range(n_slices):
                sl_size = slice_base + (1 if sl < slice_rem else 0)
                slices.append(shard_idx[s_off : s_off + sl_size])
                s_off += sl_size
            self.slice_indices.append(slices)
            offset += shard_size

        self._idx_to_location = {}
        for s_id, slices in enumerate(self.slice_indices):
            for sl_id, indices in enumerate(slices):
                for idx in indices:
                    self._idx_to_location[idx] = (s_id, sl_id)

    def get_shard(self, shard_id):
        return [self.texts[i] for i in self.shard_indices[shard_id]]

    def get_shard_indices(self, shard_id):
        return self.shard_indices[shard_id]

    def find_data_point(self, global_idx):
        if global_idx not in self._idx_to_location:
            raise ValueError(f"Index {global_idx} not found.")
        return self._idx_to_location[global_idx]

    def summary(self):
        print(f"SISADataManager: {self.n_total} docs, "
              f"{self.n_shards} shards, {self.n_slices} slices/shard")
        for s in range(self.n_shards):
            sizes = [len(sl) for sl in self.slice_indices[s]]
            print(f"  Shard {s}: {len(self.shard_indices[s])} docs "
                  f"(slices: {sizes})")


class SISAUnlearner:
    """
    Handles unlearning cost estimation via checkpoint-based retraining.
    Expected cost: shard_training_flops * (S+1)/(2S) ≈ shard/2.
    """

    def __init__(self, data_manager: SISADataManager):
        self.dm = data_manager

    def estimate_unlearning_cost(self, global_idx, shard_training_flops):
        shard_id, slice_id = self.dm.find_data_point(global_idx)
        n_slices = self.dm.n_slices
        retrain_fraction = (n_slices - slice_id) / n_slices
        exact_cost = shard_training_flops * retrain_fraction
        expected_cost = shard_training_flops * (n_slices + 1) / (2 * n_slices)
        return {
            "global_idx": global_idx, "shard_id": shard_id,
            "slice_id": slice_id, "retrain_fraction": retrain_fraction,
            "exact_cost_flops": exact_cost,
            "expected_cost_flops": expected_cost,
        }

    def expected_unlearning_cost(self, shard_training_flops):
        n_slices = self.dm.n_slices
        return shard_training_flops * (n_slices + 1) / (2 * n_slices)


class SISAEnsemble:
    """
    n-SISA inference: aggregate n shard models.
    Cost = n_shards × single_model_inference_flops.
    """

    def __init__(self, n_shards):
        self.n_shards = n_shards

    def inference_cost(self, single_model_inference_flops):
        return self.n_shards * single_model_inference_flops


# ── Holistic Cost Evaluator (Definition 4.1) ──────────────────

class HolisticCostEvaluator:
    """
    C(M) = (U_M - U_1SISA) / (I_1SISA - I_M)
    Higher C(M) = better.
    """

    def __init__(self):
        self._methods = {}

    def add_method(self, name, U, I):
        self._methods[name] = {"U": float(U), "I": float(I)}

    def compute_C(self, U_M, I_M, U_sisa, I_sisa):
        denom = I_sisa - I_M
        if abs(denom) < 1.0:
            return float("inf")
        return (U_M - U_sisa) / denom

    def compare_all(self):
        if "1-SISA" not in self._methods:
            raise ValueError("Must register '1-SISA' first.")
        sisa = self._methods["1-SISA"]
        rows = []
        for name, costs in self._methods.items():
            C = self.compute_C(costs["U"], costs["I"], sisa["U"], sisa["I"])
            rows.append({
                "method": name,
                "unlearning_flops": costs["U"],
                "inference_flops":  costs["I"],
                "holistic_C":       C,
            })
        return (
            pd.DataFrame(rows)
            .sort_values("holistic_C", ascending=False)
            .reset_index(drop=True)
        )

    def print_report(self):
        df = self.compare_all()
        print("\n" + "=" * 80)
        print("  HOLISTIC UNLEARNING COST  (Def 4.1 — Muresanu et al. 2025)")
        print("  Higher C(M) = better.  C(M) = inferences per unlearning")
        print("  request before method M becomes >= 1-SISA cost.")
        print("=" * 80)
        print(f"  {'Method':<22} {'C(M)':>12}  {'U (FLOPs)':>16}  "
              f"{'I (FLOPs)':>16}")
        print("  " + "-" * 72)
        for _, row in df.iterrows():
            c_str = (f"{row['holistic_C']:,.0f}"
                     if not np.isinf(row['holistic_C']) else "inf")
            print(f"  {row['method']:<22} {c_str:>12}  "
                  f"{row['unlearning_flops']:>16.2e}  "
                  f"{row['inference_flops']:>16.2e}")
        print("=" * 80 + "\n")


print("[OK] All core classes defined:")
print("  QuantizedKMeans, ERASE, SISADataManager, SISAUnlearner,")
print("  SISAEnsemble, HolisticCostEvaluator")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 7: Profile 1-SISA Baseline (Training + Inference)
#
#  The 1-SISA baseline is the reference for the holistic cost
#  formula C(M).  We need:
#    U_1SISA = total training FLOPs / 2  (checkpoint assumption)
#    I_1SISA = inference FLOPs for a plain query (no ICL context)
#
#  Training FLOPs are profiled with real data from the RWKU
#  Knowledge Base, then scaled to the full dataset.
# ═══════════════════════════════════════════════════════════════════

from torch.utils.flop_counter import FlopCounterMode

print("="*60)
print("  STEP 1: Profile 1-SISA TRAINING cost")
print("  Using real RWKU Knowledge Base passages")
print("="*60)

# ── Prepare a profiling batch from real RWKU passages ──────────
profile_batch_enc = tokenizer(
    all_passages[:CONFIG["profile_batch_size"]],
    padding="max_length",
    truncation=True,
    max_length=CONFIG["max_seq_length"],
    return_tensors="pt",
)
profile_batch = {k: v.to(device) for k, v in profile_batch_enc.items()}
profile_batch["labels"] = profile_batch["input_ids"].clone()

print(f"  Batch text (100 chars): {all_passages[0][:100]}...")
print(f"  Batch tokens: {profile_batch['input_ids'].shape}")

# ── Enable gradients + gradient checkpointing for memory ───────
for p in model.parameters():
    p.requires_grad_(True)
model.gradient_checkpointing_enable()
model.train()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ── Profile N training steps (forward + backward) ─────────────
train_step_flops_list = []
train_losses = []
print(f"\nProfiling {CONFIG['n_profile_steps']} training steps "
      f"(forward + backward)...")

for step_i in range(CONFIG["n_profile_steps"]):
    flop_counter = FlopCounterMode(display=False)
    with flop_counter:
        outputs = model(**profile_batch)
        loss = outputs.loss
        loss.backward()

    step_flops = flop_counter.get_total_flops()
    train_step_flops_list.append(step_flops)
    train_losses.append(loss.item())
    print(f"  Step {step_i+1}: loss={loss.item():.4f}  "
          f"FLOPs={step_flops:,.0f}")

    model.zero_grad(set_to_none=True)

# ── Disable gradients + checkpointing, return to eval mode ────
model.gradient_checkpointing_disable()
for p in model.parameters():
    p.requires_grad_(False)
model.eval()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# ── Scale to full dataset ─────────────────────────────────────
avg_train_step_flops = float(np.mean(train_step_flops_list))
total_train_samples  = len(all_passages)
total_train_steps    = (
    total_train_samples / CONFIG["profile_batch_size"]
) * CONFIG["n_epochs"]
total_training_flops = avg_train_step_flops * total_train_steps
U_1SISA = total_training_flops / 2  # checkpoint assumption

print(f"\n--- 1-SISA Training Cost ---")
print(f"  Knowledge Base size:   {total_train_samples:,} passages")
print(f"  Avg FLOPs/step:        {avg_train_step_flops:>20,.0f}")
print(f"  Total steps (scaled):  {total_train_steps:>20,.0f}")
print(f"  Total training FLOPs:  {total_training_flops:>20,.0f}")
print(f"  U_1SISA (half):        {U_1SISA:>20,.0f}")

# ══════════════════════════════════════════════════════════════
print(f"\n{'='*60}")
print("  STEP 2: Profile 1-SISA INFERENCE cost")
print("  Using real RWKU evaluation query")
print("="*60)

# 1-SISA inference = fine-tuned model on plain query (no ICL)
sample_query = all_queries[0]  # Use a real RWKU aligned query
print(f"  Query: \"{sample_query}\"")

query_enc = tokenizer(
    sample_query,
    return_tensors="pt",
    truncation=True,
    max_length=CONFIG["max_seq_length"],
)
query_inputs = {k: v.to(device) for k, v in query_enc.items()}

flop_counter = FlopCounterMode(display=False)
with flop_counter:
    with torch.no_grad():
        _ = model(**query_inputs)

I_1SISA = float(flop_counter.get_total_flops())

print(f"  Query length:    {query_inputs['input_ids'].shape[1]} tokens")
print(f"  I_1SISA FLOPs:   {I_1SISA:>20,.0f}")

print(f"\n{'='*60}")
print(f"  1-SISA Baseline Summary")
print(f"  U_1SISA = {U_1SISA:.4e}")
print(f"  I_1SISA = {I_1SISA:.4e}")
print(f"{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 8: Profile SISA {1,2,3,4}-shard Variants
#
#  For each n_shards:
#    1. Partition RWKU Knowledge Base into n shards
#    2. Compute shard training cost (scaled from profiled FLOPs)
#    3. Profile actual retraining FLOPs for one shard (isolated)
#    4. Compute unlearning cost (checkpoint assumption)
#    5. Compute inference cost (n × single model)
# ═══════════════════════════════════════════════════════════════════

print("="*60)
print("  Profiling SISA {1,2,3,4}-shard Variants")
print("  Using real RWKU Knowledge Base for shard training")
print("="*60)

sisa_costs = {}  # n_shards -> {"U": ..., "I": ..., ...}

for n_shards in CONFIG["n_shards_list"]:
    print(f"\n--- {n_shards}-SISA ---")

    # ── 1. Partition data ─────────────────────────────────────
    dm = SISADataManager(
        texts=all_passages,
        n_shards=n_shards,
        n_slices=CONFIG["n_slices_per_shard"],
    )
    dm.summary()

    # ── 2. Compute shard training cost ────────────────────────
    shard_size = len(dm.get_shard_indices(0))  # use first shard
    total_steps_per_shard = (
        shard_size / CONFIG["profile_batch_size"]
    ) * CONFIG["n_epochs"]
    shard_training_flops = avg_train_step_flops * total_steps_per_shard

    # ── 3. Profile actual retraining FLOPs for isolated shard ─
    # Use real shard data for the profiling batch
    shard_texts = dm.get_shard(0)
    shard_batch_enc = tokenizer(
        shard_texts[:CONFIG["profile_batch_size"]],
        padding="max_length",
        truncation=True,
        max_length=CONFIG["max_seq_length"],
        return_tensors="pt",
    )
    shard_batch = {k: v.to(device) for k, v in shard_batch_enc.items()}
    shard_batch["labels"] = shard_batch["input_ids"].clone()

    # Profile shard retraining (1 epoch over shard)
    for p in model.parameters():
        p.requires_grad_(True)
    model.gradient_checkpointing_enable()
    model.train()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    shard_retrain_flops = []
    for step_i in range(min(CONFIG["n_profile_steps"], 3)):
        flop_counter = FlopCounterMode(display=False)
        with flop_counter:
            outputs = model(**shard_batch)
            loss = outputs.loss
            loss.backward()
        shard_retrain_flops.append(flop_counter.get_total_flops())
        model.zero_grad(set_to_none=True)

    model.gradient_checkpointing_disable()
    for p in model.parameters():
        p.requires_grad_(False)
    model.eval()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    avg_shard_step_flops = float(np.mean(shard_retrain_flops))
    # Scale retraining to full shard (1 epoch)
    shard_retrain_total = avg_shard_step_flops * total_steps_per_shard

    # ── 4. Compute unlearning cost ────────────────────────────
    unlearner = SISAUnlearner(dm)
    U_nSISA = unlearner.expected_unlearning_cost(shard_retrain_total)

    # ── 5. Compute inference cost ─────────────────────────────
    # Profile inference with real RWKU query
    query_enc_sisa = tokenizer(
        sample_query,
        return_tensors="pt",
        truncation=True,
        max_length=CONFIG["max_seq_length"],
    )
    query_inputs_sisa = {k: v.to(device) for k, v in query_enc_sisa.items()}

    flop_counter = FlopCounterMode(display=False)
    with flop_counter:
        with torch.no_grad():
            _ = model(**query_inputs_sisa)

    single_inference_flops = float(flop_counter.get_total_flops())
    ensemble = SISAEnsemble(n_shards=n_shards)
    I_nSISA = ensemble.inference_cost(single_inference_flops)

    sisa_costs[n_shards] = {
        "U": U_nSISA,
        "I": I_nSISA,
        "shard_size": shard_size,
        "shard_training_flops": shard_retrain_total,
        "total_steps_per_shard": total_steps_per_shard,
        "single_inference_flops": single_inference_flops,
    }

    print(f"  Shard size:             {shard_size:,}")
    print(f"  Steps/shard:            {total_steps_per_shard:,.0f}")
    print(f"  Shard retrain FLOPs:    {shard_retrain_total:.4e}")
    print(f"  U_{n_shards}SISA (expected): {U_nSISA:.4e}")
    print(f"  I_{n_shards}SISA:            {I_nSISA:.4e}")

print(f"\n{'='*60}")
print("  All SISA Variants Profiled")
print(f"{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 9: Profile ERASE {1,2,3,4}-shot Variants
#
#  For each k in {1, 2, 3, 4}:
#    1. Fit ERASE with k clusters on RWKU Knowledge Base
#    2. Build ICL prompt with k examples + real RWKU query
#    3. Profile inference FLOPs -> I_ERASE_k  (sampled over 50% subjects)
#    4. ACTUALLY unlearn ONE passage per subject for 50% of all
#       available subjects, recording the REAL path (FREE/SWAP/REFIT)
#       and aggregating FLOP stats across all unlearning requests.
# ═══════════════════════════════════════════════════════════════════

import math

print("="*60)
print("  Profiling ERASE {1,2,3,4}-shot Variants")
print("  Unlearning across 50% of all available subjects")
print("="*60)

erase_results = {}  # k -> {"U": ..., "I": ..., ...}

# ── Determine the 50% subject subset ──────────────────────────
n_total_subjects  = len(subjects)
n_forget_subjects = math.ceil(n_total_subjects * 0.50)
forget_subjects   = subjects[:n_forget_subjects]   # first 50% (deterministic)

# For each forget subject, pick its first passage index as the
# unlearning target (one unlearning request per subject).
forget_indices = []
for subj in forget_subjects:
    passages_for_subj = subject_to_passages[subj]
    if passages_for_subj:
        forget_indices.append(passages_for_subj[0]["idx"])

print(f"\n  Total subjects in KB:      {n_total_subjects}")
print(f"  Subjects selected (50%):   {n_forget_subjects}")
print(f"  Unlearning requests total: {len(forget_indices)}")
print(f"  Sample forget subjects:    {forget_subjects[:3]}")
print(f"  Sample forget indices:     {forget_indices[:5]}")

# ── Pre-compute SBERT embeddings once (shared across all k) ────
print("\n[ERASE] Pre-computing Sentence-BERT embeddings for all passages...")
sbert_encoder = SentenceTransformer(CONFIG["encoder_name"])
all_embeddings = sbert_encoder.encode(
    all_passages, show_progress_bar=True, convert_to_numpy=True
).astype(np.float64)
print(f"[ERASE] Embeddings shape: {all_embeddings.shape}")

for k in CONFIG["k_shot_list"]:
    print(f"\n{'='*60}")
    print(f"  ERASE-{k}shot  |  {len(forget_indices)} unlearning requests")
    print(f"{'='*60}")

    # ── 1. Fit ERASE ──────────────────────────────────────────
    erase_k = ERASE(
        n_examples=k,
        epsilon=CONFIG["epsilon"],
        encoder_name=CONFIG["encoder_name"],
        max_iter=CONFIG["max_kmeans_iter"],
    )
    erase_k.fit(all_passages, precomputed_embeddings=all_embeddings)

    # ── 2. Profile inference FLOPs (average over 50%-subject queries) ─
    # Sample one query per forget subject for realistic inference cost.
    subject_to_queries = defaultdict(list)
    for d in aligned_data:
        subject_to_queries[d["subject"]].append(d["query"])

    inference_flops_list = []
    for subj in forget_subjects:
        q_list = subject_to_queries.get(subj, [])
        icl_query = q_list[0] if q_list else all_queries[0]
        icl_prompt = erase_k.get_icl_prompt(icl_query)

        icl_enc = tokenizer(
            icl_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048,
        )
        icl_inputs = {k_: v.to(device) for k_, v in icl_enc.items()}

        flop_counter = FlopCounterMode(display=False)
        with flop_counter:
            with torch.no_grad():
                _ = model(**icl_inputs)
        inference_flops_list.append(float(flop_counter.get_total_flops()))

    I_erase_k   = float(np.mean(inference_flops_list))
    prompt_tokens = icl_inputs["input_ids"].shape[1]   # last sampled prompt

    print(f"  Inference FLOPs profiled over {len(inference_flops_list)} queries")
    print(f"  Mean I_ERASE_{k}: {I_erase_k:>20,.0f} FLOPs")
    print(f"  Std  I_ERASE_{k}: {float(np.std(inference_flops_list)):>20,.0f} FLOPs")

    # ── 3. Unlearn 50% of subjects — one passage per subject ──
    ul_records    = []   # list of UnlearningResult
    path_counts   = {"FREE": 0, "SWAP": 0, "REFIT": 0}
    skipped       = 0

    print(f"\n  Running {len(forget_indices)} unlearning requests...")
    for req_num, orig_idx in enumerate(tqdm(forget_indices,
                                            desc=f"  ERASE-{k}shot unlearn")):
        # Skip if already removed by a prior unlearn in this loop
        if orig_idx >= len(erase_k.texts_) or not erase_k.texts_[orig_idx]:
            skipped += 1
            continue
        try:
            result = erase_k.unlearn(orig_idx)
            ul_records.append(result)
            if result.refit_needed:
                path_counts["REFIT"] += 1
            elif result.was_selected:
                path_counts["SWAP"]  += 1
            else:
                path_counts["FREE"]  += 1
        except ValueError as e:
            skipped += 1

    # ── 4. Aggregate unlearning FLOP statistics ────────────────
    if ul_records:
        ul_flops_arr  = np.array([r.unlearning_flops for r in ul_records],
                                 dtype=np.float64)
        ul_times_arr  = np.array([r.wall_time_s * 1000 for r in ul_records])
        U_erase_k     = float(ul_flops_arr.mean())   # mean cost per request
        U_total       = float(ul_flops_arr.sum())
        U_std         = float(ul_flops_arr.std())
        wall_mean_ms  = float(ul_times_arr.mean())
        wall_total_s  = float(ul_times_arr.sum() / 1000)
    else:
        U_erase_k = U_total = U_std = wall_mean_ms = wall_total_s = 0.0

    n_completed = len(ul_records)
    n_free   = path_counts["FREE"]
    n_swap   = path_counts["SWAP"]
    n_refit  = path_counts["REFIT"]

    # ── 5. Store results ───────────────────────────────────────
    erase_results[k] = {
        "U":                  U_erase_k,          # mean FLOPs/request (for C(M))
        "I":                  I_erase_k,
        "U_total_flops":      U_total,
        "U_std_flops":        U_std,
        "prompt_tokens":      prompt_tokens,
        "n_unlearn_requests": n_completed,
        "n_subjects_targeted": n_forget_subjects,
        "n_skipped":          skipped,
        "path_counts":        path_counts,
        "pct_free":           100 * n_free  / max(n_completed, 1),
        "pct_swap":           100 * n_swap  / max(n_completed, 1),
        "pct_refit":          100 * n_refit / max(n_completed, 1),
        "wall_mean_ms":       wall_mean_ms,
        "wall_total_s":       wall_total_s,
    }

    # ── 6. Print per-k summary ─────────────────────────────────
    print(f"\n  ── ERASE-{k}shot Unlearning Summary ──")
    print(f"  Subjects targeted (50%):  {n_forget_subjects}")
    print(f"  Requests completed:       {n_completed}")
    print(f"  Skipped (already gone):   {skipped}")
    print(f"")
    print(f"  Path breakdown:")
    print(f"    FREE  (O(d))     : {n_free:4d}  ({100*n_free/max(n_completed,1):5.1f}%)")
    print(f"    SWAP  (O(d))     : {n_swap:4d}  ({100*n_swap/max(n_completed,1):5.1f}%)")
    print(f"    REFIT (O(nkd·t)) : {n_refit:4d}  ({100*n_refit/max(n_completed,1):5.1f}%)")
    print(f"")
    print(f"  Unlearning FLOPs (per request):")
    print(f"    Mean  U_ERASE_{k}: {U_erase_k:>20,.0f}")
    print(f"    Std   U_ERASE_{k}: {U_std:>20,.0f}")
    print(f"    Total U_ERASE_{k}: {U_total:>20,.0f}")
    print(f"")
    print(f"  Inference FLOPs (per query):")
    print(f"    Mean  I_ERASE_{k}: {I_erase_k:>20,.0f}")
    print(f"")
    print(f"  Wall time:")
    print(f"    Mean per request : {wall_mean_ms:>10.2f} ms")
    print(f"    Total            : {wall_total_s:>10.2f} s")

print(f"\n{'='*60}")
print("  ERASE Profiling Complete (50%-subject unlearning)")
print(f"{'='*60}")

# ── Cross-k path distribution summary ─────────────────────────
print("\n  Cross-k Unlearning Path Summary:")
print(f"  {'k-shot':<12} {'Requests':>10} {'FREE%':>8} {'SWAP%':>8} {'REFIT%':>8} {'Mean U FLOPs':>16}")
print("  " + "-"*66)
for k, res in erase_results.items():
    print(f"  ERASE-{k}shot  "
          f"{res['n_unlearn_requests']:>10}  "
          f"{res['pct_free']:>7.1f}%  "
          f"{res['pct_swap']:>7.1f}%  "
          f"{res['pct_refit']:>7.1f}%  "
          f"{res['U']:>16,.0f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 10: Holistic Cost Table (Definition 4.1)
#
#  Assembles all profiled values and computes C(M) for each
#  method (both SISA and ERASE variants) against 1-SISA baseline.
#
#  C(M) = (U_M - U_1SISA) / (I_1SISA - I_M)
#
#  Higher C(M) = better (more inferences per unlearning request
#  before M becomes more expensive than 1-SISA).
# ═══════════════════════════════════════════════════════════════════

evaluator = HolisticCostEvaluator()

# ── Register SISA variants ────────────────────────────────────
for n_shards, costs in sisa_costs.items():
    evaluator.add_method(
        f"{n_shards}-SISA",
        U=costs["U"],
        I=costs["I"],
    )

# ── Register ERASE variants ───────────────────────────────────
for k, res in erase_results.items():
    evaluator.add_method(f"ERASE-{k}shot", U=res["U"], I=res["I"])

# ── Print the unified report ──────────────────────────────────
evaluator.print_report()

# ── Full DataFrame ────────────────────────────────────────────
df_holistic = evaluator.compare_all()
print("\nFull results table:")
print(df_holistic.drop(columns=["better_than_1SISA"], errors="ignore").to_string(index=False))

# ── Interpretation ────────────────────────────────────────────
print("\n" + "="*60)
print("  INTERPRETATION")
print("="*60)
print("\n  SISA methods trade off unlearning vs inference:")
print("    - More shards -> cheaper unlearning (smaller shard to retrain)")
print("    - More shards -> more expensive inference (n models)")
print("\n  ERASE methods trade off in the opposite direction:")
print("    - Unlearning ≈ free (just update k-means index)")
print("    - Inference more expensive (k ICL examples in context)")

for _, row in df_holistic.iterrows():
    if row["method"] == "1-SISA":
        continue
    c = row['holistic_C']
    if np.isinf(c):
        print(f"\n  {row['method']}: Always cheaper than 1-SISA (C=inf)")
    elif c > 0:
        print(f"\n  {row['method']}: Cheaper than 1-SISA if < {c:,.0f}"
              f" inferences per unlearning request")
        print(f"    U = {row['unlearning_flops']:.2e}")
        print(f"    I = {row['inference_flops']:.2e}")
    else:
        print(f"\n  {row['method']}: More expensive than 1-SISA (C={c:,.0f})")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 11: Detailed Results & Export
#
#  Saves all profiled data to JSON and CSV.
#  Shows ERASE unlearning path breakdown (50%-subject requests).
# ═══════════════════════════════════════════════════════════════════

export = {
    "experiment": "Unified ERASE + SISA Holistic Cost (Operator-Level FLOPs, 50%-subject unlearning)",
    "paper_erase": "Muresanu et al., ICML 2025",
    "paper_sisa":  "Bourtoule et al., 2021",
    "model":       CONFIG["model_name"],
    "dataset":     CONFIG["dataset_name"],
    "knowledge_base_size":    len(all_passages),
    "evaluation_queries":     len(all_queries),
    "unique_subjects":        len(subjects),
    "subjects_forgotten_50pct": n_forget_subjects,
    "unlearning_requests_per_erase_variant": len(forget_indices),
    "n_profile_steps":  CONFIG["n_profile_steps"],
    "max_seq_length":   CONFIG["max_seq_length"],
    "profiler": "torch.utils.flop_counter.FlopCounterMode",
    "profiling": {
        "avg_train_step_flops":   avg_train_step_flops,
        "train_step_flops_all":   train_step_flops_list,
    },
    "sisa_variants":  {},
    "erase_variants": {},
    "holistic_costs": {},
}

# ── SISA details ──────────────────────────────────────────────
for n_shards, costs in sisa_costs.items():
    export["sisa_variants"][f"{n_shards}-SISA"] = {
        "n_shards":              n_shards,
        "shard_size":            costs["shard_size"],
        "shard_training_flops":  costs["shard_training_flops"],
        "U_flops":               costs["U"],
        "I_flops":               costs["I"],
        "single_inference_flops": costs["single_inference_flops"],
        "total_steps_per_shard": costs["total_steps_per_shard"],
    }

# ── ERASE details (50%-subject aggregated stats) ───────────────
for k, res in erase_results.items():
    export["erase_variants"][f"ERASE-{k}shot"] = {
        "k":                       k,
        "U_mean_flops":            res["U"],
        "U_total_flops":           res["U_total_flops"],
        "U_std_flops":             res["U_std_flops"],
        "I_mean_flops":            res["I"],
        "prompt_tokens":           res["prompt_tokens"],
        "n_unlearn_requests":      res["n_unlearn_requests"],
        "n_subjects_targeted":     res["n_subjects_targeted"],
        "n_skipped":               res["n_skipped"],
        "path_counts":             res["path_counts"],
        "pct_free":                res["pct_free"],
        "pct_swap":                res["pct_swap"],
        "pct_refit":               res["pct_refit"],
        "wall_mean_ms":            res["wall_mean_ms"],
        "wall_total_s":            res["wall_total_s"],
    }

# ── Holistic costs ────────────────────────────────────────────
for _, row in df_holistic.iterrows():
    c_val = row["holistic_C"]
    export["holistic_costs"][row["method"]] = (
        "inf" if np.isinf(c_val) else float(c_val)
    )

# ── Save JSON ─────────────────────────────────────────────────
out_json = Path("unified_holistic_results.json")
with open(out_json, "w") as f:
    json.dump(export, f, indent=2, default=str)
print(f"Results saved to: {out_json.resolve()}")

# ── Save CSV (holistic table) ─────────────────────────────────
out_csv = Path("unified_holistic_results.csv")
df_holistic.to_csv(out_csv, index=False)
print(f"Table saved to:   {out_csv.resolve()}")

# ── Save ERASE path breakdown as separate CSV ─────────────────
path_rows = []
for k, res in erase_results.items():
    path_rows.append({
        "method":            f"ERASE-{k}shot",
        "n_requests":        res["n_unlearn_requests"],
        "n_subjects_50pct":  res["n_subjects_targeted"],
        "FREE_count":        res["path_counts"]["FREE"],
        "SWAP_count":        res["path_counts"]["SWAP"],
        "REFIT_count":       res["path_counts"]["REFIT"],
        "pct_free":          res["pct_free"],
        "pct_swap":          res["pct_swap"],
        "pct_refit":         res["pct_refit"],
        "U_mean_flops":      res["U"],
        "U_total_flops":     res["U_total_flops"],
        "U_std_flops":       res["U_std_flops"],
        "I_mean_flops":      res["I"],
        "wall_mean_ms":      res["wall_mean_ms"],
        "wall_total_s":      res["wall_total_s"],
    })
df_paths = pd.DataFrame(path_rows)
out_paths_csv = Path("erase_unlearn_path_breakdown.csv")
df_paths.to_csv(out_paths_csv, index=False)
print(f"Path breakdown:   {out_paths_csv.resolve()}")

# ── Print final summary ───────────────────────────────────────
print("\n" + "="*70)
print("  UNIFIED ERASE + SISA HOLISTIC COST EXPERIMENT COMPLETE")
print("="*70)
print(f"  Model:    {CONFIG['model_name']}")
print(f"  Dataset:  {CONFIG['dataset_name']}")
print(f"  KB size:  {len(all_passages):,} passages")
print(f"  Queries:  {len(all_queries):,} aligned queries")
print(f"  Subjects: {len(subjects)}  (forgot 50% = {n_forget_subjects})")
print(f"  Profiler: FlopCounterMode (operator-level)")
print("\n  SISA variants profiled:  " +
      ", ".join(f"{n}-SISA" for n in CONFIG["n_shards_list"]))
print("  ERASE variants profiled: " +
      ", ".join(f"ERASE-{k}shot" for k in CONFIG["k_shot_list"]))
print(f"\n  ERASE unlearning requests per variant: {len(forget_indices)}")
print("\n  ERASE Path Breakdown (50%-subject unlearning):")
print(f"  {'Method':<14} {'Requests':>9} {'FREE':>7} {'SWAP':>7} {'REFIT':>7} {'Mean U (FLOPs)':>16}")
print("  " + "-"*64)
for k, res in erase_results.items():
    print(f"  ERASE-{k}shot   "
          f"{res['n_unlearn_requests']:>9}  "
          f"{res['pct_free']:>6.1f}%  "
          f"{res['pct_swap']:>6.1f}%  "
          f"{res['pct_refit']:>6.1f}%  "
          f"{res['U']:>16,.0f}")
print(f"\n  Output files:")
print(f"    {out_json}")
print(f"    {out_csv}")
print(f"    {out_paths_csv}")
print("="*70)
